In [0]:
dim_table = dbutils.widgets.get("dim_table")
raw_table = dbutils.widgets.get("raw_table")
date_table = dbutils.widgets.get("date_table")
payer_table = dbutils.widgets.get("payer_table")
office_table = dbutils.widgets.get("office_table")
source_system_table = dbutils.widgets.get("source_system_table")
client_table = dbutils.widgets.get("client_table")
payment_type_table = dbutils.widgets.get("payment_type_table")
payment_detail_table = dbutils.widgets.get("payment_detail_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW dim_src AS
  SELECT
    ss.SourceSystemKey                          AS source_system_key,
    oc.OfficeKey                                AS office_key,
    coalesce(p.PayerKey, -1)                    AS payor_key,
    coalesce(cl.ClientKey, -1)                  AS client_key,
    coalesce(pd.DateKey, -1)                    AS posted_date_key,
    coalesce(dd.DateKey, -1)                    AS deposit_date_key,
    de.DateKey                                  AS date_entered_key,
    ca.invoice_number                           AS invoice_number,
    ca.cash_collected                           AS cash_collected,
    coalesce(pt.PaymentTypeKey, -1)             AS payment_type_key,
    pl.PaymentDetailKey                         AS payment_detail_key,
    ca.payment_number                           AS payment_number,
    ca._load_timestamp                          AS _load_timestamp,
    ca._file_name                               AS _file_name
  FROM {raw_table} ca
  LEFT JOIN {date_table} pd
      ON pd.CalendarDate = ca.posted_date
  LEFT JOIN {date_table} dd
      ON dd.CalendarDate = ca.deposit_date
  LEFT JOIN {date_table} de
      ON de.CalendarDate = ca.date_entered
  LEFT JOIN {payer_table} p
      ON p.PayerID = ca.payor_id
  LEFT JOIN {office_table} oc
      ON oc.OfficeNumber = ca.office_number
  LEFT JOIN {source_system_table} ss
      ON ss.SourceSystemName = ca.source_system
  LEFT JOIN {client_table} cl
      ON cl.SourceSystemId = ca.client_number
      AND cl.OfficeNumber = ca.office_number
      AND cl.SourceSystem = 'BEARS'
  LEFT JOIN {payment_type_table} pt
      ON coalesce(pt.PaymentTypeDescription,'') = coalesce(ca.Product,'')
  LEFT JOIN {payment_detail_table} pl
      ON coalesce(pl.BatchID,'') = coalesce(ca.batch_id,'')
      AND coalesce(pl.CheckID,'') = coalesce(ca.check_id,'')
      AND coalesce(pl.Type,'') = coalesce(ca.Type,'')
      AND coalesce(pl.Bank,'') = coalesce(ca.Bank,'')
      AND coalesce(pl.BatchNumber,'') = coalesce(ca.batch_number,'')
      AND pl.SourceSystem = 'BEARS'
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {dim_table} tgt
USING (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY
                       invoice_number,
                       cash_collected,
                       payment_number, 
                       date_entered_key
                   ORDER BY _load_timestamp DESC
               ) AS rn
        FROM dim_src
    ) sub
    WHERE rn = 1
) src
ON tgt.invoice_number   = src.invoice_number
AND tgt.cash_collected   = src.cash_collected
AND tgt.payment_number   = src.payment_number
AND tgt.date_entered_key  = src.date_entered_key

WHEN NOT MATCHED THEN
INSERT (
    source_system_key,
    office_key,
    payor_key,
    client_key,
    posted_date_key,
    deposit_date_key,
    date_entered_key,
    invoice_number,
    cash_collected,
    payment_type_key,
    payment_detail_key,
    payment_number,
    _load_timestamp,
    _file_name 
)
VALUES (
    src.source_system_key,
    src.office_key,
    src.payor_key,
    src.client_key,
    src.posted_date_key,
    src.deposit_date_key,
    src.date_entered_key,
    src.invoice_number,
    src.cash_collected,
    src.payment_type_key,
    src.payment_detail_key,
    src.payment_number,
    src._load_timestamp,
    src._file_name 
)
""")
)